# Tutorial 120 — Expand retrieval deterministically

[Tutorial 110](./110_hybrid_ranking.ipynb) combined semantic, lexical, and
code-aware rankings with ordinary Reciprocal Rank Fusion (RRF). This tutorial
leaves those RRF scores unchanged and adds a separate
**deterministic expansion** step for exact repository targets
and relevant code-name matches.

The expansion can guarantee context for an exact file, an exact function, or
direct code relationships. These are guarantees added after ranking; they are
not alternative retrieval scores or hidden routing weights.

The diagram below summarizes the complete path from question to final evidence.
[Tutorial 130](./130_project_chat.ipynb) will place this retrieval pipeline
inside a complete persistent Project Chat session.

**Before you begin:** See the [tutorial guide](./README.md) for setup, model downloads, learning paths, and workspace safety.


![KíLín retrieval pipeline](./120_context_expansion.svg)


## 1. Prepare KíLín

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "kilin").is_dir():
    root = root.parent

print(f"KíLín root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import kilin


## 2. Load the embedding model

The semantic channel uses the canonical embedding model for the backend selected
by KíLín. The lexical and code-aware channels do not need this model, but all
three will operate on the same repository chunks.

**Model files:** These cells do not download weights by default. Cache the embedding model first (Tutorial 20) or deliberately set `ALLOW_DOWNLOAD = True`. Loading can require substantial memory.


In [ ]:
REQUESTED_BACKEND = kilin.BACKEND_AUTO

SELECTED_BACKEND = kilin.choose_backend(REQUESTED_BACKEND)

MODEL_ID = (
    kilin.DEFAULT_EMBEDDING_MODEL_MLX
    if SELECTED_BACKEND == "mlx"
    else kilin.DEFAULT_EMBEDDING_MODEL_TORCH
)

print(f"Requested backend: {REQUESTED_BACKEND}")
print(f"Resolved backend: {SELECTED_BACKEND}")
print(f"Embedding model: {MODEL_ID}")


In [ ]:
ALLOW_DOWNLOAD = False  # Opt in to network downloads.


In [ ]:
embedding_session = kilin.EmbeddingSession(
    MODEL_ID,
    backend=SELECTED_BACKEND,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 3. Prepare repository data

Prepare the three retrieval structures introduced in Tutorials
[70](./70_semantic_retrieval.ipynb),
[80](./80_lexical_retrieval.ipynb), and
[90](./90_code_aware_retrieval.ipynb): semantic retrieval captures meaning,
lexical retrieval captures wording and identifiers, and code-aware retrieval
captures source relationships.

For a shorter first run, index `kilin/retrievals` and `kilin/rag.py`, which cover the functions in the examples. Change `include_paths` to `["kilin"]` to index the entire package; this uses more time and memory.


In [ ]:
report = kilin.scan_repository(
    root,
    extensions=[".py"],
    include_paths=["kilin/retrievals", "kilin/rag.py"],
)


In [ ]:
source_files = kilin.read_repository_files(report)
units = kilin.extract_source_units(source_files)

print(f"Source files: {len(source_files)}")
print(f"Natural units: {len(units)}")


In [ ]:
chunks = kilin.chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


In [ ]:
semantic_index = kilin.build_semantic_index(
    chunks, embedding_session, batch_size=16
)
lexical_index = kilin.build_lexical_index(chunks)
code_graph = kilin.build_code_graph(source_files, units)


## 4. Use questions that isolate deterministic signals

Four questions make the expansion rules visible:

- **Q1** contains no exact target but may match words in
  function names;
- **Q2** names an exact file;
- **Q3** names an exact function;
- **Q4** names an exact function and asks about its repository relationships.

Using controlled questions lets us verify each deterministic rule separately.


In [ ]:
QUESTIONS = {
    "Q1": "What does semantic search mean?",
    "Q2": "Walk me through kilin/retrievals/strategies/retrieval_semantic.py",
    "Q3": "What does semantic_search do?",
    "Q4": "How is semantic_search used across the repository?",
}

for label, question in QUESTIONS.items():
    print(f"{label}: {question}")


## 5. Detect deterministic signals

`plan_context_expansion()` uses deterministic repository information and no LLM. It
records context that should be guaranteed later; it does not rank chunks and it
does not modify RRF scores.


In [ ]:
plans = {
    label: kilin.plan_context_expansion(question, code_graph)
    for label, question in QUESTIONS.items()
}


In [ ]:
for label, plan in plans.items():
    print(f"\n{'=' * 64}")
    print(f"{label}: {QUESTIONS[label]}")
    print(f"Signals: {', '.join(plan.signals)}")

    paths = [path.as_posix() for path in plan.paths]
    print(
        "Exact files: "
        + (", ".join(paths) if paths else "none")
    )

    print(
        "Exact functions: "
        + (
            ", ".join(plan.functions)
            if plan.functions
            else "none"
        )
    )

    print(
        "Relationship: "
        + ("yes" if plan.relationship else "no")
    )

    print(
        "Distance-1 functions: "
        + (
            ", ".join(plan.related_functions)
            if plan.related_functions
            else "none"
        )
    )


The expected signals for the bundled source snapshot are:

| Question | Deterministic expansion |
| --- | --- |
| Q1 | code-name word match (for example, `semantic_search`) |
| Q2 | all available chunks from the named source file |
| Q3 | named function and bounded callee context |
| Q4 | named function and bounded direct code relationships |

The additions depend on the code graph, available chunks, and expansion
budgets. They supplement the normal retrieval ranking without changing its
scores. The exact related functions may change as the repository evolves.


## 6. Run the three retrieval channels

Every question still passes through semantic, lexical, and code-aware
retrieval. Deterministic planning does not bypass or reweight these searches.


In [ ]:
INSTRUCTION = (
    "Given a question about a software repository, retrieve relevant code "
    "and documentation that help answer the question."
)


In [ ]:
channel_results = {}

for label, question in QUESTIONS.items():
    semantic_query = kilin.format_qwen3_query(
        question=question,
        instruction=INSTRUCTION,
    )

    channel_results[label] = {
        "semantic": kilin.semantic_search(
            semantic_index,
            semantic_query,
            embedding_session,
            count=15,
        ),
        "lexical": kilin.lexical_search(
            lexical_index,
            question,
            count=15,
        ),
        "code": kilin.code_search(
            code_graph,
            chunks,
            question,
            count=15,
            max_distance=2,
        ),
    }


## 7. Combine the rankings with ordinary RRF

This is the same rank fusion used in
[Tutorial 110](./110_hybrid_ranking.ipynb). At this point the result is the
ordinary top-N hybrid evidence; no exact-file, exact-function, or relationship
guarantee has been added yet.


In [ ]:
RRF_COUNT = 5
rrf_results = {}

for label, results in channel_results.items():
    rrf_results[label] = kilin.hybrid_search(
        semantic=results["semantic"],
        lexical=results["lexical"],
        code=results["code"],
        count=RRF_COUNT,
    )


In [ ]:
for label, results in rrf_results.items():
    print(f"\n{'=' * 64}")
    print(f"{label}: {QUESTIONS[label]}")
    print(f"Normal RRF top {RRF_COUNT}:")

    for rank, result in enumerate(results, start=1):
        name = result.chunk.symbol or result.chunk.path.name
        print(
            f"{rank}. {result.score:.4f} "
            f"{result.chunk.path}::{name}"
        )


## 8. Expand the RRF evidence

`expand_retrieval_context()` preserves the normal RRF results first and then
appends any context guaranteed by the deterministic plan: all chunks from an
exact file, all chunks from an exact function, or chunks from directly related
functions. Duplicate chunks are removed.


In [ ]:
expanded_results = {
    label: kilin.expand_retrieval_context(
        rrf_results[label],
        chunks,
        code_graph,
        plans[label],
    )
    for label in QUESTIONS
}


## 9. Inspect what deterministic expansion added

Compare the original RRF evidence with the expanded evidence. Keeping these two
sets visible makes it possible to verify that deterministic expansion adds context
rather than silently changing the ranking itself.


In [ ]:
for label, expanded in expanded_results.items():
    rrf_ids = {
        result.chunk.identifier
        for result in rrf_results[label]
    }
    added = [
        chunk
        for chunk in expanded
        if chunk.identifier not in rrf_ids
    ]

    print(f"\n{'=' * 64}")
    print(f"{label}: {QUESTIONS[label]}")
    print(f"Signals: {', '.join(plans[label].signals)}")
    print(f"RRF chunks: {len(rrf_results[label])}")
    print(f"Added chunks: {len(added)}")
    print(f"Final evidence chunks: {len(expanded)}")

    if not added:
        print("Added context: none")
        continue

    print("Added context:")
    for chunk in added:
        name = chunk.symbol or chunk.path.name
        print(
            f"  {chunk.path}::{name} "
            f"lines {chunk.start_line}-{chunk.end_line}"
        )


## 10. Verify exact-file expansion

For Q2, every searchable chunk from the requested file should be
present in the final evidence, even if it was not in the RRF top five.


In [ ]:
q2_plan = plans["Q2"]
q2_paths = set(q2_plan.paths)

expected_q2 = {
    chunk.identifier
    for chunk in chunks
    if chunk.path in q2_paths
}
actual_q2 = {
    chunk.identifier
    for chunk in expanded_results["Q2"]
}

print(f"Requested file chunks: {len(expected_q2)}")
print(
    "All requested file chunks included: "
    f"{expected_q2 <= actual_q2}"
)


## 11. Verify exact-function expansion

For Q3, every chunk belonging to `semantic_search` should be included.
If the function fits in one natural chunk, this will normally be one
complete function chunk.


In [ ]:
q3_plan = plans["Q3"]
q3_names = {
    name.rsplit(".", 1)[-1]
    for name in q3_plan.functions
}

q3_function_chunks = [
    chunk
    for chunk in expanded_results["Q3"]
    if chunk.symbol in q3_names
]

print(f"Matched functions: {q3_plan.functions}")
print(f"Function chunks included: {len(q3_function_chunks)}")

for chunk in q3_function_chunks:
    print(
        f"{chunk.path} "
        f"lines {chunk.start_line}-{chunk.end_line}"
    )
    print()
    print(chunk.text)


## 12. Verify relationship expansion

For Q4, the exact function is kept and direct function neighbors are
added from both directions of the code graph.

For a `calls` relationship this means direct callers and direct
callees. Other direct function-to-function relationships are also
eligible when they exist in the graph.


In [ ]:
q4_plan = plans["Q4"]
targets = set(q4_plan.functions)

print(f"Target functions: {q4_plan.functions}")
print(f"Distance-1 functions: {q4_plan.related_functions}")
print()

for relation in code_graph.relations:
    touches_target = (
        relation.source in targets
        or relation.target in targets
    )
    if touches_target:
        print(
            f"{relation.source} "
            f"--{relation.relation}--> "
            f"{relation.target}"
        )


## 13. Inspect the final evidence

The final evidence starts with the normal RRF top results and then contains any
additional guaranteed context. This ordered evidence set is what a later
`RAGSession` can insert into the answer-generating LLM prompt.


In [ ]:
for label, expanded in expanded_results.items():
    print(f"\n{'=' * 64}")
    print(f"{label}: {QUESTIONS[label]}")
    print(f"Final evidence chunks: {len(expanded)}")

    for rank, chunk in enumerate(expanded, start=1):
        name = chunk.symbol or chunk.path.name
        print(
            f"{rank}. {chunk.path}::{name} "
            f"lines {chunk.start_line}-{chunk.end_line}"
        )


## 14. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Continue the tutorial series

**Previous:** [Tutorial 110 — Combine retrieval channels](./110_hybrid_ranking.ipynb)

**Next:** [Tutorial 130 — Chat with a KíLín project](./130_project_chat.ipynb)
